<a href="https://colab.research.google.com/github/PauloRadatz/opendss-python-examples/blob/main/presentations/ieee_et_pes_pels_joint_chapter_workshop/03_finding_lowest_voltage_bus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 — Finding the lowest-voltage bus

Here is a small task that sounds trivial: after solving the power flow, find the bus with the lowest voltage on the feeder.

Doing this in OpenDSS by hand means opening the *Show Voltages LN Nodes* report, scrolling through hundreds of rows, and finding the minimum. On `ckt5` that is roughly 3,000 nodes. Nobody does that.

In Python it is two lines.

## Setup

In [ ]:
!pip install py-dss-interface
!git clone https://github.com/PauloRadatz/opendss-python-examples.git
%cd opendss-python-examples

In [ ]:
import pathlib
import pandas as pd
import py_dss_interface

start_dir = pathlib.Path.cwd().resolve()
repo_root = next(
    parent for parent in [start_dir, *start_dir.parents]
    if (parent / "feeder_models").exists()
)
dss_file = repo_root / "feeder_models" / "EPRITestCircuits" / "ckt5" / "Master_ckt5.dss"

## Compile and solve at peak load

Lowest voltages typically appear under heavy load, so we use the peak condition.

In [ ]:
dss = py_dss_interface.DSS()
dss.text(f"compile [{dss_file}]")
dss.text("set loadmult=1.0")
dss.text("solve")

print(f"Converged: {dss.solution.converged}")

## The two-line answer

`dss.circuit.buses_vmag_pu` returns voltage magnitudes in per unit for every node, and `dss.circuit.nodes_names` returns the names in the same order. We pair them and pick the minimum.

In [ ]:
voltages_pu = dss.circuit.buses_vmag_pu
node_names = dss.circuit.nodes_names

min_v = min(voltages_pu)
min_node = node_names[voltages_pu.index(min_v)]
min_bus = min_node.split(".")[0]

print(f"Lowest voltage on the feeder")
print(f"  Bus.Node : {min_node}")
print(f"  Bus only : {min_bus}")
print(f"  Voltage  : {min_v:.4f} pu")

## A bit more context: the 10 lowest-voltage nodes

Once the data is in Python, expanding the question is cheap. The cell below builds a sorted DataFrame and shows the 10 worst nodes.

In [ ]:
voltages_df = pd.DataFrame({
    "Node": node_names,
    "Voltage (pu)": voltages_pu,
})

# Drop nodes with zero voltage (they typically correspond to unused phases on single-phase laterals).
voltages_df = voltages_df[voltages_df["Voltage (pu)"] > 0]

voltages_df = voltages_df.sort_values("Voltage (pu)").reset_index(drop=True)
voltages_df.head(10)

## How many nodes are below 0.95 pu?

On a real feeder under stress this is the question that usually matters. One line of pandas.

In [ ]:
below_limit = voltages_df[voltages_df["Voltage (pu)"] < 0.95]
print(f"Nodes below 0.95 pu: {len(below_limit)} out of {len(voltages_df)}")

## Key takeaways

- A task that is tedious in the OpenDSS standalone — searching a long voltage report for a minimum — is two lines in Python.
- `dss.circuit.buses_vmag_pu` and `dss.circuit.nodes_names` are the simplest building blocks for any voltage-screening study.
- Once the result is in a DataFrame, asking follow-up questions (top 10 worst nodes, count of nodes below a limit, group by feeder section) is essentially free.

This is what we mean by *Python turns OpenDSS into a programmable engineering platform*. The simulation engine does not change. What changes is how easily we can ask questions of the result.

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [pauloradatz.me](https://www.pauloradatz.me)
- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)